Run before everything

In [2]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


Part1. ZIGGIE

In [ ]:
import time
import math

# =============================================================================
# PART 1 - COLOR RECOGNITION + APRILTAG + PICKUP
# UGot Mecanum Robot | Competition Code
# =============================================================================

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
CENTER_X         = 320
GAP_TOLERANCE    = 10
STRICT_TOLERANCE = 2.5
TARGET_DISTANCE  = 0.16

HSV_RED_RANGES = [
    (0,   15,  50, 255, 50, 255),
    (155, 179, 50, 255, 50, 255),
    (0,   20,  40, 255, 40, 255),
    (150, 179, 40, 255, 40, 255),
]
HSV_GREEN_RANGES = [
    (35,  90,  40, 255, 40, 255),
    (25,  95,  30, 255, 30, 255),
]

COLOR_PIXEL_THRESHOLD = 0.002
COLOR_CONFIRM_FRAMES  = 1

RED_GROUP = [
    "Red", "Orange", "Pink", "Maroon", "Crimson",
    "Rose", "Magenta", "Coral", "Scarlet", "Tomato",
    "Salmon", "Burgundy", "Ruby", "Vermillion", "Brick"
]
GREEN_GROUP = [
    "Green", "Lime", "Teal", "Olive", "Cyan",
    "Mint", "Yellow-Green", "Forest", "Emerald", "Jade",
    "Chartreuse", "Sage", "Moss", "Hunter", "Fern"
]

COLOR_POLL_DELAY = 0.05


# ==========================================
# HSV COLOR DETECTION
# ==========================================
def _check_hsv_color(frame, ranges):
    try:
        import cv2
        import numpy as np
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        total_pixels = hsv.shape[0] * hsv.shape[1]
        matched = 0
        for (h_min, h_max, s_min, s_max, v_min, v_max) in ranges:
            lower = np.array([h_min, s_min, v_min], dtype=np.uint8)
            upper = np.array([h_max, s_max, v_max], dtype=np.uint8)
            mask  = cv2.inRange(hsv, lower, upper)
            matched += int(cv2.countNonZero(mask))
        ratio = matched / total_pixels
        return ratio >= COLOR_PIXEL_THRESHOLD, ratio
    except:
        return False, 0.0


def detect_color_in_frame(frame, target):
    ranges = HSV_RED_RANGES if target == "Red" else HSV_GREEN_RANGES
    return _check_hsv_color(frame, ranges)


def get_camera_frame():
    frame = None
    for fn_name in ["get_camera_frame", "get_image_frame", "get_frame",
                    "read_camera_frame", "capture_frame", "get_video_frame"]:
        fn = getattr(got, fn_name, None)
        if fn:
            try:
                result = fn()
                if result is not None:
                    import numpy as np
                    arr = np.array(result)
                    if arr.ndim == 3 and arr.shape[2] in (3, 4):
                        frame = arr
                        break
            except:
                pass
    if frame is None:
        for fn_name in ["get_color_image", "get_rgb_image", "get_bgr_image"]:
            fn = getattr(got, fn_name, None)
            if fn:
                try:
                    result = fn()
                    if result is not None:
                        import numpy as np
                        arr = np.array(result)
                        if arr.ndim == 3:
                            frame = arr
                            break
                except:
                    pass
    return frame


# ==========================================
# HELPERS
# ==========================================
def safe_get_tag():
    try:
        data = got.get_apriltag_total_info()
        if data and len(data) > 0:
            return data
    except:
        return None
    return None


def extract_color_name(raw):
    if not raw:
        return None
    if isinstance(raw, str):
        return raw
    if isinstance(raw, int):
        return str(raw)
    if isinstance(raw, (list, tuple)):
        if len(raw) == 0:
            return None
        first = raw[0]
        if isinstance(first, str):
            return first
        if isinstance(first, int):
            return str(first)
        if isinstance(first, dict):
            return str(first.get("name") or first.get("color") or first.get("label") or "")
    return str(raw)


def scan_all_colors(raw):
    if not raw:
        return None
    entries = raw if isinstance(raw, (list, tuple)) else [raw]
    for entry in entries:
        if isinstance(entry, str):
            name = entry
        elif isinstance(entry, int):
            name = str(entry)
        elif isinstance(entry, dict):
            name = str(entry.get("name") or entry.get("color") or entry.get("label") or "")
        else:
            name = str(entry)
        if name in RED_GROUP:
            return "Red"
        if name in GREEN_GROUP:
            return "Green"
    return None


def is_target_color(raw, target):
    return scan_all_colors(raw) == target


def start_color_recognition():
    for fn_name in [
        "open_color_recognition", "start_color_recognition",
        "enable_color_recognition", "color_recognition_open",
        "set_color_recognition", "open_color_detect",
        "start_color_detect", "enable_color_detect",
    ]:
        fn = getattr(got, fn_name, None)
        if fn:
            try:
                fn()
                time.sleep(0.5)
            except:
                pass
    try:
        got.open_camera()
    except:
        pass


def wait_for_color(target):
    got.show_light_rgb([0, 1, 2, 3], 255, 255, 255)
    start_color_recognition()
    time.sleep(1.5)

    confirm_streak = 0
    none_count     = 0

    while True:
        frame = get_camera_frame()
        hsv_match = False
        if frame is not None:
            hsv_match, _ = detect_color_in_frame(frame, target)
            none_count = 0
        else:
            none_count += 1
            if none_count % 20 == 0:
                try:
                    got.close_camera()
                    time.sleep(0.3)
                    got.open_camera()
                    time.sleep(1.0)
                    start_color_recognition()
                    time.sleep(0.5)
                except:
                    pass

        sdk_match = False
        try:
            cam_data  = got.get_color_total_info()
            sdk_match = is_target_color(cam_data, target)
        except:
            pass

        matched = hsv_match or sdk_match

        if matched:
            confirm_streak += 1
        else:
            confirm_streak = 0

        if confirm_streak >= COLOR_CONFIRM_FRAMES:
            # ── JUDGE PRINT: colour recognised ──────────────────────────
            print(f"[TASK {1 if target == 'Red' else 2}] {target.upper()} COLOUR RECOGNISED")
            got.show_light_rgb([0, 1, 2, 3], 0, 0, 0)
            return True

        time.sleep(COLOR_POLL_DELAY)


def show_color_screen(color_name):
    if color_name == "Red":
        got.screen_display_background(3)
    elif color_name == "Green":
        got.screen_display_background(6)
    time.sleep(2)
    got.screen_display_background(0)


# ==========================================
# APRILTAG HUNT + ALIGNMENT
# ==========================================
def AP_advanced_hunt(fwd_speed=4, strafe_speed=8):
    half_time  = 3
    full_time  = 6
    start_time = time.time()

    # Search & lock
    while True:
        AP_info = safe_get_tag()

        if AP_info:
            tag_x = AP_info[0][1]
            # ── JUDGE PRINT: April tag recognised ───────────────────────
            print(f"[TASK 3] APRIL TAG RECOGNISED — tag_x={tag_x}, dist={AP_info[0][6]:.3f}m")
            if tag_x < 245:
                got.mecanum_move_xyz(-10, fwd_speed, 0)
                continue
            elif tag_x > 395:
                got.mecanum_move_xyz(10, fwd_speed, 0)
                continue
            else:
                got.mecanum_stop()
                time.sleep(0.3)
                break

        elapsed = time.time() - start_time
        if elapsed < half_time:
            current_strafe = strafe_speed
        elif (elapsed - half_time) % (full_time * 2) < full_time:
            current_strafe = -strafe_speed
        else:
            current_strafe = strafe_speed

        got.mecanum_move_xyz(int(current_strafe), fwd_speed, 0)
        time.sleep(0.02)

    # Alignment & approach
    while True:
        AP_info = safe_get_tag()
        if not AP_info:
            got.mecanum_move_xyz(4, 0, 0)
            time.sleep(0.1)
            continue

        AP_x    = AP_info[0][1]
        dist    = AP_info[0][6]
        error_x = AP_x - CENTER_X

        current_tol = STRICT_TOLERANCE if dist < (TARGET_DISTANCE + 0.05) else GAP_TOLERANCE

        if abs(error_x) > current_tol:
            # Proportional strafe: scales from 5 (small error) up to 25 (large error)
            # Clamped so it never overshoots at high speed
            raw_speed  = int(abs(error_x) * 0.12)          # ~5 at 40px, ~19 at 160px
            side_speed = max(5, min(raw_speed, 25))
            side_move  = side_speed if error_x > 0 else -side_speed

            # Claw rotation assist: kick in when error is large (far left/right)
            # Helps sweep the tag into frame faster before body catches up
            if abs(error_x) > 80:
                claw_yaw = int(max(min(error_x * 0.15, 30), -30))
                got.mecanum_move_xyz(int(side_move), 0, claw_yaw)
            else:
                fwd_move = 2 if dist > TARGET_DISTANCE else 0
                got.mecanum_move_xyz(int(side_move), fwd_move, 0)
        else:
            if dist > TARGET_DISTANCE:
                if dist > 1.0:
                    current_speed = 50
                elif dist > 0.5:
                    current_speed = 40
                elif dist > 0.3:
                    current_speed = 30
                else:
                    current_speed = 5
                got.mecanum_move_xyz(0, current_speed, 0)
            else:
                got.mecanum_stop()
                break

        time.sleep(0.05)


# ==========================================
# PICKUP SEQUENCE
# ==========================================
def advanced_pick_up():
    got.mechanical_single_joint_control(joint=2, angle=15, duration=500)
    got.mechanical_single_joint_control(joint=3, angle=-75, duration=500)
    got.mechanical_clamp_release()
    time.sleep(1.0)

    AP_info = safe_get_tag()
    if not AP_info:
        joint1, joint3 = 0, 10
    else:
        AP_x        = AP_info[0][1]
        AP_distance = AP_info[0][6]
        joint1 = int(max(min((AP_x - CENTER_X) * -0.12, 90), -90))
        joint3 = int(max(min(AP_distance * 100 - 85, 90), -90))

    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1)

    got.mecanum_translate_speed_times(angle=0, speed=2, times=2, unit=1)
    time.sleep(1)

    got.mechanical_clamp_close()
    time.sleep(1.0)

    got.mechanical_joint_control(0, 10, -20, 1000)

    # ── JUDGE PRINT: pickup complete ────────────────────────────────────
    print("[TASK 4] TOKEN PICKED UP — clamp closed, token held off ground")


# ==========================================
# MAIN
# ==========================================
def run_autonomous_mission():
    try:
        wait_for_color("Red")
        show_color_screen("Red")

        wait_for_color("Green")
        show_color_screen("Green")
        

        AP_advanced_hunt()
        advanced_pick_up()

    except KeyboardInterrupt:
        got.mecanum_stop()
    except Exception as e:
        print(f"[ERROR] {e}")
        got.mecanum_stop()


if __name__ == "__main__":
    run_autonomous_mission()

[TASK 1] RED COLOUR RECOGNISED
[TASK 2] GREEN COLOUR RECOGNISED
[TASK 3] APRIL TAG RECOGNISED — tag_x=295.59, dist=1.400m


In [8]:
import time
import math

# =============================================================================
# PART 1 - COLOR RECOGNITION + APRILTAG + PICKUP
# UGot Mecanum Robot | Competition Code
# =============================================================================

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
CENTER_X         = 320
GAP_TOLERANCE    = 10
STRICT_TOLERANCE = 2.5
TARGET_DISTANCE  = 0.16

HSV_RED_RANGES = [
    (0,   15,  50, 255, 50, 255),
    (155, 179, 50, 255, 50, 255),
    (0,   20,  40, 255, 40, 255),
    (150, 179, 40, 255, 40, 255),
]
HSV_GREEN_RANGES = [
    (35,  90,  40, 255, 40, 255),
    (25,  95,  30, 255, 30, 255),
]

COLOR_PIXEL_THRESHOLD = 0.002
COLOR_CONFIRM_FRAMES  = 1

RED_GROUP = [
    "Red", "Orange", "Pink", "Maroon", "Crimson",
    "Rose", "Magenta", "Coral", "Scarlet", "Tomato",
    "Salmon", "Burgundy", "Ruby", "Vermillion", "Brick"
]
GREEN_GROUP = [
    "Green", "Lime", "Teal", "Olive", "Cyan",
    "Mint", "Yellow-Green", "Forest", "Emerald", "Jade",
    "Chartreuse", "Sage", "Moss", "Hunter", "Fern"
]

COLOR_POLL_DELAY = 0.05


# ==========================================
# HSV COLOR DETECTION
# ==========================================
def _check_hsv_color(frame, ranges):
    try:
        import cv2
        import numpy as np
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        total_pixels = hsv.shape[0] * hsv.shape[1]
        matched = 0
        for (h_min, h_max, s_min, s_max, v_min, v_max) in ranges:
            lower = np.array([h_min, s_min, v_min], dtype=np.uint8)
            upper = np.array([h_max, s_max, v_max], dtype=np.uint8)
            mask  = cv2.inRange(hsv, lower, upper)
            matched += int(cv2.countNonZero(mask))
        ratio = matched / total_pixels
        return ratio >= COLOR_PIXEL_THRESHOLD, ratio
    except:
        return False, 0.0


def detect_color_in_frame(frame, target):
    ranges = HSV_RED_RANGES if target == "Red" else HSV_GREEN_RANGES
    return _check_hsv_color(frame, ranges)


def get_camera_frame():
    frame = None
    for fn_name in ["get_camera_frame", "get_image_frame", "get_frame",
                    "read_camera_frame", "capture_frame", "get_video_frame"]:
        fn = getattr(got, fn_name, None)
        if fn:
            try:
                result = fn()
                if result is not None:
                    import numpy as np
                    arr = np.array(result)
                    if arr.ndim == 3 and arr.shape[2] in (3, 4):
                        frame = arr
                        break
            except:
                pass
    if frame is None:
        for fn_name in ["get_color_image", "get_rgb_image", "get_bgr_image"]:
            fn = getattr(got, fn_name, None)
            if fn:
                try:
                    result = fn()
                    if result is not None:
                        import numpy as np
                        arr = np.array(result)
                        if arr.ndim == 3:
                            frame = arr
                            break
                except:
                    pass
    return frame


# ==========================================
# HELPERS
# ==========================================
def safe_get_tag():
    try:
        data = got.get_apriltag_total_info()
        if data and len(data) > 0:
            return data
    except:
        return None
    return None


def extract_color_name(raw):
    if not raw:
        return None
    if isinstance(raw, str):
        return raw
    if isinstance(raw, int):
        return str(raw)
    if isinstance(raw, (list, tuple)):
        if len(raw) == 0:
            return None
        first = raw[0]
        if isinstance(first, str):
            return first
        if isinstance(first, int):
            return str(first)
        if isinstance(first, dict):
            return str(first.get("name") or first.get("color") or first.get("label") or "")
    return str(raw)


def scan_all_colors(raw):
    if not raw:
        return None
    entries = raw if isinstance(raw, (list, tuple)) else [raw]
    for entry in entries:
        if isinstance(entry, str):
            name = entry
        elif isinstance(entry, int):
            name = str(entry)
        elif isinstance(entry, dict):
            name = str(entry.get("name") or entry.get("color") or entry.get("label") or "")
        else:
            name = str(entry)
        if name in RED_GROUP:
            return "Red"
        if name in GREEN_GROUP:
            return "Green"
    return None


def is_target_color(raw, target):
    return scan_all_colors(raw) == target


def start_color_recognition():
    for fn_name in [
        "open_color_recognition", "start_color_recognition",
        "enable_color_recognition", "color_recognition_open",
        "set_color_recognition", "open_color_detect",
        "start_color_detect", "enable_color_detect",
    ]:
        fn = getattr(got, fn_name, None)
        if fn:
            try:
                fn()
                time.sleep(0.5)
            except:
                pass
    try:
        got.open_camera()
    except:
        pass


def wait_for_color(target):
    got.show_light_rgb([0, 1, 2, 3], 255, 255, 255)
    start_color_recognition()
    time.sleep(1.5)

    confirm_streak = 0
    none_count     = 0

    while True:
        frame = get_camera_frame()
        hsv_match = False
        if frame is not None:
            hsv_match, _ = detect_color_in_frame(frame, target)
            none_count = 0
        else:
            none_count += 1
            if none_count % 20 == 0:
                try:
                    got.close_camera()
                    time.sleep(0.3)
                    got.open_camera()
                    time.sleep(1.0)
                    start_color_recognition()
                    time.sleep(0.5)
                except:
                    pass

        sdk_match = False
        try:
            cam_data  = got.get_color_total_info()
            sdk_match = is_target_color(cam_data, target)
        except:
            pass

        matched = hsv_match or sdk_match

        if matched:
            confirm_streak += 1
        else:
            confirm_streak = 0

        if confirm_streak >= COLOR_CONFIRM_FRAMES:
            # ── JUDGE PRINT: colour recognised ──────────────────────────
            print(f"[TASK {1 if target == 'Red' else 2}] {target.upper()} COLOUR RECOGNISED")
            got.show_light_rgb([0, 1, 2, 3], 0, 0, 0)
            return True

        time.sleep(COLOR_POLL_DELAY)


def show_color_screen(color_name):
    if color_name == "Red":
        got.screen_display_background(3)
    elif color_name == "Green":
        got.screen_display_background(6)
    time.sleep(2)
    got.screen_display_background(0)


# ==========================================
# APRILTAG HUNT + ALIGNMENT
# ==========================================
def AP_advanced_hunt(fwd_speed=4, strafe_speed=8):
    half_time  = 3
    full_time  = 6
    start_time = time.time()

    # Once we've seen the tag, use narrow zigzag + forward creep when lost
    tag_seen_before = False
    narrow_strafe   = 4   # smaller left/right when tag was previously seen
    narrow_fwd      = 2   # slow creep forward during narrow recovery search
    narrow_dir      = 1   # flip direction each time tag is lost in alignment

    # Search & lock
    while True:
        AP_info = safe_get_tag()

        if AP_info:
            tag_x = AP_info[0][1]
            tag_seen_before = True

            # ── JUDGE PRINT: April tag recognised ───────────────────────
            print(f"[TASK 3] APRIL TAG RECOGNISED — tag_x={tag_x}, dist={AP_info[0][6]:.3f}m")

            if tag_x < 245:
                got.mecanum_move_xyz(-10, fwd_speed, 0)
                continue
            elif tag_x > 395:
                got.mecanum_move_xyz(10, fwd_speed, 0)
                continue
            else:
                got.mecanum_stop()
                time.sleep(0.3)
                break

        # Tag not visible — choose search mode based on history
        if not tag_seen_before:
            # Wide zigzag: never seen tag yet
            elapsed = time.time() - start_time
            if elapsed < half_time:
                current_strafe = strafe_speed
            elif (elapsed - half_time) % (full_time * 2) < full_time:
                current_strafe = -strafe_speed
            else:
                current_strafe = strafe_speed
            got.mecanum_move_xyz(int(current_strafe), fwd_speed, 0)
        else:
            # Narrow zigzag + forward creep: tag was seen but now lost
            elapsed = time.time() - start_time
            if elapsed % (full_time * 2) < full_time:
                current_strafe = narrow_strafe
            else:
                current_strafe = -narrow_strafe
            got.mecanum_move_xyz(int(current_strafe), narrow_fwd, 0)

        time.sleep(0.02)

    # Alignment & approach
    while True:
        AP_info = safe_get_tag()
        if not AP_info:
            # Lost tag mid-alignment — narrow oscillate to recover, creep forward
            got.mecanum_move_xyz(int(narrow_strafe * narrow_dir), narrow_fwd, 0)
            narrow_dir = -narrow_dir  # flip each miss so it wiggles back and forth
            time.sleep(0.1)
            continue

        AP_x    = AP_info[0][1]
        dist    = AP_info[0][6]
        error_x = AP_x - CENTER_X

        current_tol = STRICT_TOLERANCE if dist < (TARGET_DISTANCE + 0.05) else GAP_TOLERANCE

        if abs(error_x) > current_tol:
            # Proportional strafe: scales from 5 (small error) up to 25 (large error)
            raw_speed  = int(abs(error_x) * 0.12)
            side_speed = max(5, min(raw_speed, 25))
            side_move  = side_speed if error_x > 0 else -side_speed

            # Claw rotation assist when far off-centre
            if abs(error_x) > 80:
                claw_yaw = int(max(min(error_x * 0.15, 30), -30))
                got.mecanum_move_xyz(int(side_move), 0, claw_yaw)
            else:
                fwd_move = 2 if dist > TARGET_DISTANCE else 0
                got.mecanum_move_xyz(int(side_move), fwd_move, 0)
        else:
            if dist > TARGET_DISTANCE:
                if dist > 1.0:
                    current_speed = 50
                elif dist > 0.5:
                    current_speed = 40
                elif dist > 0.3:
                    current_speed = 30
                else:
                    current_speed = 5
                got.mecanum_move_xyz(0, current_speed, 0)
            else:
                got.mecanum_stop()
                break

        time.sleep(0.05)


# ==========================================
# PICKUP SEQUENCE
# ==========================================
def advanced_pick_up():
    got.mechanical_single_joint_control(joint=2, angle=15, duration=500)
    got.mechanical_single_joint_control(joint=3, angle=-75, duration=500)
    got.mechanical_clamp_release()
    time.sleep(1.0)

    AP_info = safe_get_tag()
    if not AP_info:
        joint1, joint3 = 0, 10
    else:
        AP_x        = AP_info[0][1]
        AP_distance = AP_info[0][6]
        joint1 = int(max(min((AP_x - CENTER_X) * -0.12, 90), -90))
        joint3 = int(max(min(AP_distance * 100 - 85, 90), -90))

    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1)

    got.mecanum_translate_speed_times(angle=0, speed=2, times=2, unit=1)
    time.sleep(1)

    got.mechanical_clamp_close()
    time.sleep(1.0)

    got.mechanical_joint_control(0, 10, -20, 1000)

    # ── JUDGE PRINT: pickup complete ────────────────────────────────────
    print("[TASK 4] TOKEN PICKED UP — clamp closed, token held off ground")


# ==========================================
# MAIN
# ==========================================
def run_autonomous_mission():
    try:
        wait_for_color("Red")
        show_color_screen("Red")

        wait_for_color("Green")
        show_color_screen("Green")

        AP_advanced_hunt()
        advanced_pick_up()

    except KeyboardInterrupt:
        got.mecanum_stop()
    except Exception as e:
        print(f"[ERROR] {e}")
        got.mecanum_stop()


if __name__ == "__main__":
    run_autonomous_mission()

[TASK 1] RED COLOUR RECOGNISED
[TASK 2] GREEN COLOUR RECOGNISED
[TASK 3] APRIL TAG RECOGNISED — tag_x=78.74, dist=1.420m
[TASK 3] APRIL TAG RECOGNISED — tag_x=78.73, dist=1.420m
[TASK 3] APRIL TAG RECOGNISED — tag_x=83.87, dist=1.390m
[TASK 3] APRIL TAG RECOGNISED — tag_x=36.61, dist=1.350m
[TASK 3] APRIL TAG RECOGNISED — tag_x=14.79, dist=1.330m
[TASK 3] APRIL TAG RECOGNISED — tag_x=19.57, dist=1.240m
[TASK 3] APRIL TAG RECOGNISED — tag_x=22.34, dist=1.240m
[TASK 3] APRIL TAG RECOGNISED — tag_x=45.89, dist=1.180m
[TASK 3] APRIL TAG RECOGNISED — tag_x=49.12, dist=1.130m
[TASK 3] APRIL TAG RECOGNISED — tag_x=57.31, dist=1.100m
[TASK 3] APRIL TAG RECOGNISED — tag_x=57.97, dist=1.100m
[TASK 3] APRIL TAG RECOGNISED — tag_x=61.25, dist=1.040m
[TASK 3] APRIL TAG RECOGNISED — tag_x=51.98, dist=1.040m
[TASK 3] APRIL TAG RECOGNISED — tag_x=47.62, dist=1.030m
[TASK 3] APRIL TAG RECOGNISED — tag_x=48.53, dist=1.020m
[TASK 3] APRIL TAG RECOGNISED — tag_x=48.19, dist=1.000m
[TASK 3] APRIL TAG RECOG